# 10 — Backpropagation: one number at a time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/010-backpropagation.ipynb)

Training means finding which way to move each number to make the loss smaller. Backpropagation is the chain rule applied carefully, and here you do it on a single weight and bias, check it against a nudge, and then break the sign.

## 1. The numbers

The model is one weight and one bias, the input is 2, and the target is 10. The loss is the squared gap between the guess and the target, so a wrong guess is punished more than proportionally.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

x = 2.0                                               # one input
w = 3.0                                               # the weight we are learning
b = -1.0                                              # the bias we are learning
y = 10.0                                              # the target the model should have produced

pred = w * x + b                                      # forward pass: the model's guess
err = pred - y                                        # how far the guess is from the target
loss = err ** 2                                       # squared error: the one number to minimise
print("prediction =", pred)                           # 5.0
print("error =", err)                                 # -5.0, the guess is too low
print("loss =", loss)                                 # 25.0

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `x = 2.0` — the single input.
- `w = 3.0` — the weight being learned.
- `b = -1.0` — the bias being learned.
- `y = 10.0` — the answer the model should produce.
- `pred = w * x + b` — the forward pass.
- `err = pred - y` — the residual, signed: negative means too low.
- `loss = err ** 2` — squaring removes the sign and punishes large errors.
- `print("prediction", ...)` — the guess.
- `print("error", ...)` — how far off, with a sign.
- `print("loss", ...)` — the number training will push down.

## 2. Step by step: the backward pass

Backpropagation walks the graph backwards, multiplying one local slope at a time. The nudge check then measures the same slope without any calculus, which is how you catch a mistake.

In [ ]:
dL_dpred = 2 * err                                     # slope of the square: 2 times the error
dL_dw = dL_dpred * x                                  # chain rule: times d(pred)/dw, which is x
dL_db = dL_dpred * 1.0                                # chain rule: d(pred)/db is 1
print(f"by hand:  dL/dw = {dL_dw:+.2f}   dL/db = {dL_db:+.2f}")   # -20 and -10

eps = 1e-4                                            # a tiny nudge
L_plus = ((w + eps) * x + b - y) ** 2                 # loss with the weight nudged up
L_minus = ((w - eps) * x + b - y) ** 2                # loss with the weight nudged down
numeric = (L_plus - L_minus) / (2 * eps)              # slope measured by the two nudges
print(f"nudged:   dL/dw = {numeric:+.2f}")            # -20, same as the hand version
print(f"difference = {abs(numeric - dL_dw):.2e}")     # ~1e-11: rounding only

**Line by line**

- `dL_dpred = 2 * err` — derivative of a square, at the current error.
- `dL_dw = dL_dpred * x` — chain rule through the multiply.
- `dL_db = dL_dpred * 1.0` — chain rule through the add; the slope is 1.
- `print(f"by hand: ...")` — the two slopes, from calculus only.
- `eps = 1e-4` — the size of the nudge.
- `L_plus` — the loss one nudge above the current weight.
- `L_minus` — the loss one nudge below it.
- `numeric = (L_plus - L_minus) / (2 * eps)` — rise over run, symmetric.
- `print(f"nudged: ...")` — the same slope, measured without calculus.
- `abs(numeric - dL_dw)` — the two answers should agree to about ten digits.

## 3. Plot: the loss, with the gradient as an arrow

The loss as a function of one weight is a parabola, and its lowest point is the weight that predicts the target exactly. The gradient is the slope under your feet; the arrow shows the step you take against it.

In [ ]:
import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab

w_grid = np.linspace(-1, 12, 200)                     # weights from -1 to 12
loss_grid = (w_grid * x + b - y) ** 2                 # the loss at every one of them

fig, ax = plt.subplots(figsize=(6, 3))                # one axis, wide and short
ax.plot(w_grid, loss_grid, color="tab:blue", label="loss(w)")   # the parabola
ax.scatter([w], [loss], color="tab:red", zorder=3, label="where we are")   # current weight
step_w = w - 0.15 * dL_dw                             # move against the slope
step_loss = (step_w * x + b - y) ** 2                 # the loss at the new weight
ax.annotate("", xy=(step_w, step_loss), xytext=(w, loss), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))   # the descent arrow
ax.axvline((y - b) / x, color="grey", linestyle="--", label="best w")   # the bottom of the parabola
ax.set_xlabel("weight w")                             # x-axis label
ax.set_ylabel("loss")                                 # y-axis label
ax.set_title("the gradient points down the parabola")  # the takeaway
ax.legend()                                           # name the marks
plt.show()                                            # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — Colab already ships the plotting library.
- `w_grid = np.linspace(-1, 12, 200)` — a range of candidate weights.
- `loss_grid = ...` — the loss for each candidate, one array.
- `plt.subplots(figsize=(6, 3))` — a figure and its single axis.
- `ax.plot(...)` — the parabola itself.
- `ax.scatter([w], [loss])` — marks the current (weight, loss) pair.
- `step_w = w - 0.15 * dL_dw` — a small step against the gradient.
- `step_loss = ...` — the loss at the end of that step.
- `ax.annotate("", xy=..., ...)` — an arrow from here to there; empty text.
- `ax.axvline((y - b) / x, ...)` — the weight that makes the error zero.
- `ax.set_xlabel("weight w")` — names the horizontal axis.
- `ax.set_ylabel("loss")` — names the vertical axis.
- `ax.set_title(...)` — the takeaway.
- `ax.legend()` — matches each mark to its label.
- `plt.show()` — draws the figure.

## 4. Interactive: put the weight anywhere

Drag the weight across the parabola. The readout gives the loss and the slope, and you can watch the slope shrink to zero as the dot reaches the bottom.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(w_value=3.0):                                # w_value comes from the slider
    p = w_value * x + b                               # forward pass at this weight
    e = p - y                                         # residual
    g = 2 * e * x                                     # d(loss)/dw at this weight
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.plot(w_grid, loss_grid, color="tab:blue")     # the same fixed parabola
    plt.scatter([w_value], [e ** 2], color="tab:red", zorder=3)   # current point
    plt.axvline((y - b) / x, color="grey", linestyle="--")   # the minimum
    plt.xlabel("weight w")                            # x-axis label
    plt.ylabel("loss")                                # y-axis label
    plt.title(f"w = {w_value:.2f} -> loss = {e ** 2:.2f}, slope = {g:+.2f}")   # live readout
    plt.show()                                        # draw it

interact(show, w_value=(-1.0, 12.0, 0.5))             # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(w_value=3.0)` — the function that redraws one frame.
- `p = ...` — a forward pass at the slider's weight.
- `e = p - y` — the residual, signed.
- `g = 2 * e * x` — the gradient, computed in one line.
- `plt.figure(...)` — a new canvas for each frame.
- `plt.plot(...)` — the same fixed parabola.
- `plt.scatter(...)` — the current point on it.
- `plt.axvline(...)` — the bottom of the parabola.
- `plt.xlabel("weight w")` — names the horizontal axis.
- `plt.ylabel("loss")` — names the vertical axis.
- `plt.title(f"...")` — loss and slope, both live.
- `plt.show()` — draws the frame.
- `interact(show, w_value=(...))` — the slider, its range and its step.

## 5. When it breaks: the wrong sign

Gradient descent says subtract the gradient. Add it instead and every step climbs the parabola, which turns a small error into a huge one. The two curves below differ only in that one sign.

In [ ]:
lr = 0.02                                             # step size, shared by both runs
w_ok, w_bad = 3.0, 3.0                                # the same starting weight
ok_hist, bad_hist = [], []                            # one loss per step

for step in range(30):                                # thirty steps each
    g_ok = 2 * (w_ok * x + b - y) * x                 # gradient for the correct run
    w_ok = w_ok - lr * g_ok                           # subtract: downhill
    ok_hist.append((w_ok * x + b - y) ** 2)           # record the loss
    g_bad = 2 * (w_bad * x + b - y) * x               # gradient for the broken run
    w_bad = w_bad + lr * g_bad                        # ADD: uphill, the classic sign slip
    bad_hist.append((w_bad * x + b - y) ** 2)         # record the loss

print(f"correct sign: w = {w_ok:.3f}  loss = {ok_hist[-1]:.4f}")   # settles near 5.5
print(f"wrong sign:   w = {w_bad:.1f}  loss = {bad_hist[-1]:.2e}")  # runs away
print(f"loss grew {bad_hist[-1] / bad_hist[0]:.0f}x in 30 steps")  # growth, not shrinkage

plt.figure(figsize=(6, 3))                            # new figure
plt.plot(ok_hist, color="tab:green", label="w = w - lr * slope")   # falls
plt.plot(bad_hist, color="tab:red", label="w = w + lr * slope")    # rises
plt.yscale("log")                                     # the two scales differ wildly
plt.xlabel("step")                                    # x-axis label
plt.ylabel("loss (log scale)")                        # y-axis label
plt.title("a sign flip turns descent into divergence")  # the failure
plt.legend()                                          # name the curves
plt.show()                                            # render

**Line by line**

- `lr = 0.02` — the step size used by both runs.
- `w_ok, w_bad = 3.0, 3.0` — identical starting points, so only the sign differs.
- `ok_hist, bad_hist = [], []` — one loss list per run.
- `for step in range(30)` — thirty steps for each rule.
- `g_ok = ...` — the gradient for the correct run.
- `w_ok = w_ok - lr * g_ok` — the correct update.
- `ok_hist.append(...)` — record its loss.
- `g_bad = ...` — the same gradient formula for the broken run.
- `w_bad = w_bad + lr * g_bad` — the broken update; `+` instead of `-`.
- `bad_hist.append(...)` — record its loss.
- `print(f"correct sign ...")` — where the good run ended up.
- `print(f"wrong sign ...")` — how far the bad run flew.
- `print(f"loss grew ...x")` — the ratio makes the divergence explicit.
- `plt.figure(...)` — a new canvas.
- `plt.plot(ok_hist, ...)` — the green curve, falling.
- `plt.plot(bad_hist, ...)` — the red curve, rising.
- `plt.yscale("log")` — a log axis, because the red curve leaves the chart.
- `plt.xlabel("step")` — names the horizontal axis.
- `plt.ylabel(...)` — notes that the axis is logged.
- `plt.title(...)` — the failure in one line.
- `plt.legend()` — names which line is which update rule.
- `plt.show()` — draws the figure.

## Your turn

In section 4, find the weight where the slope reads 0.00 and compare it with `(y - b) / x`. Then set `lr = 0.05` in section 5 and predict how many steps the correct run needs to get within 0.01 of the bottom.